## Another Example: Logistic Regression

Let's return to the diabetes classification example from earlier in the course. We'll try to predict whether a patient has diabetes from BMI alone. Each patient is a point $(x_i, y_i)$ in $\mathbb{R}^2$, where $x_i$ is BMI and $y_i$ is either 0 or 1.

In [7]:
diabetes_df = pd.read_csv("../04_linear_independence/hyperplane-ex/data/diabetes.csv")

bmi = diabetes_df["BMI"].to_numpy(dtype=float)
diabetes_outcome = diabetes_df["Outcome"].to_numpy(dtype=float)

mask_no_diabetes = diabetes_outcome == 0
mask_yes_diabetes = diabetes_outcome == 1

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=bmi[mask_no_diabetes],
        y=diabetes_outcome[mask_no_diabetes],
        mode="markers",
        marker=dict(size=8, color="orange", opacity=0.55),
        hovertemplate="BMI = %{x:.1f}<br>Outcome = 0<extra></extra>",
        name="No diabetes",
    )
)
fig.add_trace(
    go.Scatter(
        x=bmi[mask_yes_diabetes],
        y=diabetes_outcome[mask_yes_diabetes],
        mode="markers",
        marker=dict(size=8, color="#3D81F6", opacity=0.55),
        hovertemplate="BMI = %{x:.1f}<br>Outcome = 1<extra></extra>",
        name="Diabetes",
    )
)

fig.update_xaxes(
    title="BMI",
    gridcolor="#f0f0f0",
    showline=True,
    linecolor="black",
    linewidth=1,
)
fig.update_yaxes(
    title="Outcome",
    tickmode="array",
    tickvals=[0, 1],
    ticktext=["0", "1"],
    range=[-0.1, 1.1],
    gridcolor="#f0f0f0",
    showline=True,
    linecolor="black",
    linewidth=1,
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=60, t=60, b=60),
    width=700,
    font=dict(family=FONT_FAMILY, color="black"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
)
fig


We'll use the logistic regression model

$$p_i(\vec w) = \sigma(w_0 + w_1 x_i) = \frac{1}{1 + e^{-(w_0 + w_1 x_i)}}$$

where $p_i(\vec w)$ is the predicted probability that patient $i$ has diabetes.

A natural average loss for this problem is the average logistic loss

$$R_\text{log}(\vec w) = -\frac{1}{n} \sum_{i=1}^n \left[y_i \log(p_i(\vec w)) + (1-y_i)\log(1-p_i(\vec w))\right].$$

Unlike least squares, this optimization problem has no closed-form solution for $\vec w^*$, so gradient descent is one of the main tools we use to find the optimal parameters.


In [8]:
X_logistic = np.column_stack([np.ones(len(bmi)), bmi])


def sigmoid(z):
    out = np.empty_like(z, dtype=float)
    positive = z >= 0
    out[positive] = 1 / (1 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    out[~positive] = exp_z / (1 + exp_z)
    return out


def average_log_loss(w):
    z = X_logistic @ w
    return np.mean(np.logaddexp(0, z) - diabetes_outcome * z)


def logistic_grad(w):
    probabilities = sigmoid(X_logistic @ w)
    return (X_logistic.T @ (probabilities - diabetes_outcome)) / len(bmi)


alpha_logistic = 0.0072

logistic_history = run_gradient_descent(
    w0=np.zeros(2),
    alpha=alpha_logistic,
    loss_fn=average_log_loss,
    grad_fn=logistic_grad,
    tol=1e-2,
    max_iter=50000,
    record_every=500,
    loss_name="average_loss",
    param_names=["w0", "w1"],
)

logistic_history = logistic_history.rename(columns={"i": "t"})

logistic_history.round(4)


t =     0 | w^(0) = [   0.000,    0.000] | average_loss =    0.6931 | ||grad|| =  3.812418
t =   500 | w^(500) = [  -0.129,   -0.010] | average_loss =    0.6619 | ||grad|| =  0.035130
t =  1000 | w^(1000) = [  -0.253,   -0.007] | average_loss =    0.6576 | ||grad|| =  0.033857
t =  1500 | w^(1500) = [  -0.373,   -0.003] | average_loss =    0.6536 | ||grad|| =  0.032628
t =  2000 | w^(2000) = [  -0.488,    0.000] | average_loss =    0.6499 | ||grad|| =  0.031443
t =  2500 | w^(2500) = [  -0.599,    0.004] | average_loss =    0.6465 | ||grad|| =  0.030301
t =  3000 | w^(3000) = [  -0.706,    0.007] | average_loss =    0.6433 | ||grad|| =  0.029203
t =  3500 | w^(3500) = [  -0.809,    0.010] | average_loss =    0.6404 | ||grad|| =  0.028146
t =  4000 | w^(4000) = [  -0.909,    0.013] | average_loss =    0.6376 | ||grad|| =  0.027129
t =  4500 | w^(4500) = [  -1.005,    0.015] | average_loss =    0.6350 | ||grad|| =  0.026153


t =  5000 | w^(5000) = [  -1.097,    0.018] | average_loss =    0.6327 | ||grad|| =  0.025215
t =  5500 | w^(5500) = [  -1.186,    0.021] | average_loss =    0.6305 | ||grad|| =  0.024314
t =  6000 | w^(6000) = [  -1.272,    0.023] | average_loss =    0.6284 | ||grad|| =  0.023450
t =  6500 | w^(6500) = [  -1.355,    0.026] | average_loss =    0.6265 | ||grad|| =  0.022619
t =  7000 | w^(7000) = [  -1.435,    0.028] | average_loss =    0.6247 | ||grad|| =  0.021823
t =  7500 | w^(7500) = [  -1.512,    0.030] | average_loss =    0.6231 | ||grad|| =  0.021058
t =  8000 | w^(8000) = [  -1.587,    0.032] | average_loss =    0.6215 | ||grad|| =  0.020324


t =  8500 | w^(8500) = [  -1.659,    0.035] | average_loss =    0.6201 | ||grad|| =  0.019620
t =  9000 | w^(9000) = [  -1.728,    0.037] | average_loss =    0.6188 | ||grad|| =  0.018944
t =  9500 | w^(9500) = [  -1.795,    0.039] | average_loss =    0.6175 | ||grad|| =  0.018295
t = 10000 | w^(10000) = [  -1.860,    0.040] | average_loss =    0.6163 | ||grad|| =  0.017672
t = 10500 | w^(10500) = [  -1.922,    0.042] | average_loss =    0.6153 | ||grad|| =  0.017073
t = 11000 | w^(11000) = [  -1.983,    0.044] | average_loss =    0.6142 | ||grad|| =  0.016499
t = 11500 | w^(11500) = [  -2.041,    0.046] | average_loss =    0.6133 | ||grad|| =  0.015946
t = 12000 | w^(12000) = [  -2.097,    0.047] | average_loss =    0.6124 | ||grad|| =  0.015416
t = 12500 | w^(12500) = [  -2.152,    0.049] | average_loss =    0.6116 | ||grad|| =  0.014906


t = 13000 | w^(13000) = [  -2.205,    0.050] | average_loss =    0.6108 | ||grad|| =  0.014415
t = 13500 | w^(13500) = [  -2.256,    0.052] | average_loss =    0.6101 | ||grad|| =  0.013944
t = 14000 | w^(14000) = [  -2.305,    0.053] | average_loss =    0.6094 | ||grad|| =  0.013490
t = 14500 | w^(14500) = [  -2.353,    0.055] | average_loss =    0.6088 | ||grad|| =  0.013054
t = 15000 | w^(15000) = [  -2.399,    0.056] | average_loss =    0.6082 | ||grad|| =  0.012634
t = 15500 | w^(15500) = [  -2.444,    0.057] | average_loss =    0.6076 | ||grad|| =  0.012230
t = 16000 | w^(16000) = [  -2.487,    0.059] | average_loss =    0.6071 | ||grad|| =  0.011840


t = 16500 | w^(16500) = [  -2.529,    0.060] | average_loss =    0.6066 | ||grad|| =  0.011465
t = 17000 | w^(17000) = [  -2.570,    0.061] | average_loss =    0.6062 | ||grad|| =  0.011104
t = 17500 | w^(17500) = [  -2.609,    0.062] | average_loss =    0.6057 | ||grad|| =  0.010756
t = 18000 | w^(18000) = [  -2.647,    0.063] | average_loss =    0.6053 | ||grad|| =  0.010420
t = 18500 | w^(18500) = [  -2.684,    0.064] | average_loss =    0.6049 | ||grad|| =  0.010097
t = 18654 | w^(18654) = [  -2.695,    0.065] | average_loss =    0.6048 | ||grad|| =  0.010000


,t,average_loss,grad_norm,w0,w1
0,0,0.6931,3.8124,0.0000,0.0000
1,500,0.6619,0.0351,-0.1292,-0.0103
2,1000,0.6576,0.0339,-0.2533,-0.0066
3,1500,0.6536,0.0326,-0.3729,-0.0031
4,2000,0.6499,0.0314,-0.4882,0.0003
5,2500,0.6465,0.0303,-0.5993,0.0036
6,3000,0.6433,0.0292,-0.7063,0.0067
7,3500,0.6404,0.0281,-0.8095,0.0097
8,4000,0.6376,0.0271,-0.9089,0.0127
9,4500,0.6350,0.0262,-1.0048,0.0155


Because we stored every iterate in `logistic_history`, we can build the same side-by-side figure. The left panel tracks the average loss, while the right panel shows the logistic curve corresponding to the selected iteration.


In [9]:
def make_logistic_gradient_descent_slider(history_df):
    history_df = history_df.reset_index(drop=True)
    bmi_curve = np.linspace(bmi.min() - 1, bmi.max() + 1, 400)

    first = history_df.iloc[0]

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Average Loss Across Iterations", "Logistic Curve for the Selected Iterate"),
        horizontal_spacing=0.14,
    )

    fig.add_trace(
        go.Scatter(
            x=history_df["t"],
            y=history_df["average_loss"],
            mode="lines+markers",
            line=dict(color="#3D81F6", width=3),
            marker=dict(size=6, color="#3D81F6"),
            hovertemplate="t = %{x}<br>average loss = %{y:.4f}<extra></extra>",
            name="Average loss",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=[first["t"]],
            y=[first["average_loss"]],
            mode="markers",
            marker=dict(size=14, color="orange", line=dict(width=1, color="black")),
            hovertemplate="t = %{x}<br>average loss = %{y:.4f}<extra></extra>",
            name="Current iterate",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=bmi[mask_no_diabetes],
            y=diabetes_outcome[mask_no_diabetes],
            mode="markers",
            marker=dict(size=8, color="orange", opacity=0.55),
            hovertemplate="BMI = %{x:.1f}<br>Outcome = 0<extra></extra>",
            name="No diabetes",
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=bmi[mask_yes_diabetes],
            y=diabetes_outcome[mask_yes_diabetes],
            mode="markers",
            marker=dict(size=8, color="#3D81F6", opacity=0.55),
            hovertemplate="BMI = %{x:.1f}<br>Outcome = 1<extra></extra>",
            name="Diabetes",
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=bmi_curve,
            y=sigmoid(first["w0"] + first["w1"] * bmi_curve),
            mode="lines",
            line=dict(color="#004D40", width=4),
            hovertemplate="BMI = %{x:.1f}<br>predicted probability = %{y:.3f}<extra></extra>",
            name="Logistic curve",
        ),
        row=1,
        col=2,
    )

    steps = []
    for row in history_df.itertuples(index=False):
        steps.append(
            dict(
                method="update",
                args=[
                    {
                        "x": [
                            history_df["t"],
                            [row.t],
                            bmi[mask_no_diabetes],
                            bmi[mask_yes_diabetes],
                            bmi_curve,
                        ],
                        "y": [
                            history_df["average_loss"],
                            [row.average_loss],
                            diabetes_outcome[mask_no_diabetes],
                            diabetes_outcome[mask_yes_diabetes],
                            sigmoid(row.w0 + row.w1 * bmi_curve),
                        ],
                    },
                    {
                        "title": {
                            "text": f"Gradient Descent for Logistic Regression | t = {row.t} | average loss = {row.average_loss:.4f}"
                        }
                    },
                ],
                label=str(int(row.t)),
            )
        )

    fig.update_xaxes(
        title="Iteration",
        gridcolor="#f0f0f0",
        showline=True,
        linecolor="black",
        linewidth=1,
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title="Average Logistic Loss",
        gridcolor="#f0f0f0",
        showline=True,
        linecolor="black",
        linewidth=1,
        row=1,
        col=1,
    )
    fig.update_xaxes(
        title="BMI",
        range=[bmi.min() - 1, bmi.max() + 1],
        gridcolor="#f0f0f0",
        showline=True,
        linecolor="black",
        linewidth=1,
        row=1,
        col=2,
    )
    fig.update_yaxes(
        title="Outcome / Predicted Probability",
        range=[-0.08, 1.08],
        tickmode="array",
        tickvals=[0, 0.5, 1],
        gridcolor="#f0f0f0",
        showline=True,
        linecolor="black",
        linewidth=1,
        row=1,
        col=2,
    )

    fig.update_layout(
        title=f"Gradient Descent for Logistic Regression | t = 0 | average loss = {first['average_loss']:.4f}",
        width=1150,
        height=500,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(family=FONT_FAMILY, color="black", size=15),
        margin=dict(l=60, r=60, t=80, b=60),
        sliders=[
            dict(
                active=0,
                currentvalue={"prefix": "Iteration: "},
                pad={"t": 35},
                steps=steps,
            )
        ],
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    )
    return fig


interactive_logistic_fig = make_logistic_gradient_descent_slider(logistic_history)
interactive_logistic_fig


:::{tip} Activity 1
:class: dropdown

Suppose we have a dataset with 3 points, $(1, 2)$, $(3, 5)$, and $(4, 6)$. Suppose we'd like to find optimal model parameters for simple linear regression using squared loss. Instead of using our closed-form formulas, we use gradient descent.

With an initial parameter vector of $\vec w^{(0)} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$ and a step size of $\alpha = 0.5$, perform one iteration of gradient descent.

:::{seealso} Solution
:class: dropdown

Here,

$$X = \begin{bmatrix} 1 & 1 \\ 1 & 3 \\ 1 & 4 \end{bmatrix}, \qquad \vec y = \begin{bmatrix} 2 \\ 5 \\ 6 \end{bmatrix}.$$

For squared loss,

$$\nabla R_\text{sq}(\vec w) = \frac{2}{n}(X^TX \vec w - X^T \vec y).$$

At $\vec w^{(0)} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$, this becomes

$$\nabla R_\text{sq}(\vec w^{(0)}) = \frac{2}{3}\left(\vec 0 - X^T \vec y\right) = \frac{2}{3}\left(-\begin{bmatrix} 13 \\ 41 \end{bmatrix}\right) = \begin{bmatrix} -\frac{26}{3} \\ -\frac{82}{3} \end{bmatrix}.$$

So one gradient descent step gives

$$\vec w^{(1)} = \vec w^{(0)} - \alpha \nabla R_\text{sq}(\vec w^{(0)}) = \begin{bmatrix} 0 \\ 0 \end{bmatrix} - 0.5 \begin{bmatrix} -\frac{26}{3} \\ -\frac{82}{3} \end{bmatrix} = \begin{bmatrix} \frac{13}{3} \\ \frac{41}{3} \end{bmatrix}.$$

So, after one iteration,

$$\vec w^{(1)} = \begin{bmatrix} \frac{13}{3} \\ \frac{41}{3} \end{bmatrix}.$$

:::
:::
